# GitHub Metrics — Exploratory Data Mining (snapshot)

**Scope:** basic data mining on a *frozen snapshot* of `data/github_metrics.csv`
(`data/github_metrics_snapshot.csv`, 5,169 repos at capture time). The collection
notebook (`github_api.ipynb`) is still running and appending to the live CSV — this
analysis reads only the snapshot, so it never interferes with the run. Re-run after a
fresh snapshot once more data has landed.

**Question under study (per the project brief):** which activity signals separate
*decaying* open-source repos from *healthy* ones? We look for: distribution shape of
each metric, decay-stage segmentation, how bus factor / contributor base tracks
inactivity, and whether a small number of repo "archetypes" emerge.

**House rules:** pandas only (no new deps — matplotlib/sklearn are not in the env);
plots are minimal hand-rolled SVG written to `figures/eda_*.svg`; derived output goes
to `data/github_metrics_snapshot_enriched.csv` so the live collection file is never
touched.

**Data provenance:** see `github_api.ipynb` for how each column is collected (search
API `total_count` over a trailing window, core API for commits/releases/contributors).

In [ ]:
import os, json, math
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.cwd()
if not (BASE / "data").is_dir():  # nbconvert sets cwd to the notebook's dir
    BASE = BASE.parent
DATA_DIR = BASE / "data"
FIG_DIR = BASE / "figures"
FIG_DIR.mkdir(exist_ok=True)

SNAP_CSV = DATA_DIR / "github_metrics_snapshot.csv"
ENRICHED_CSV = DATA_DIR / "github_metrics_snapshot_enriched.csv"

df = pd.read_csv(SNAP_CSV)
print(f"snapshot: {len(df):,} repos  (captured from the live collection mid-run)")
df.head()

In [ ]:
# --- minimal SVG plotting (no matplotlib in this env) -------------------------
def _svg_open(w, h):
    return [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" '
            f'viewBox="0 0 {w} {h}" font-family="monospace" font-size="11">']

def _axis(x0, y0, x1, y1):
    return [f'<line x1="{x0}" y1="{y0}" x2="{x1}" y2="{y0}" stroke="#888"/>',
            f'<line x1="{x0}" y1="{y0}" x2="{x0}" y2="{y1}" stroke="#888"/>']

def svg_hist(series, path, title="", bins=40, log_scale=True, color="#4c8bf5"):
    s = series.dropna()
    if log_scale:
        s = np.log1p(s.clip(lower=0))
    lo, hi = float(s.min()), float(s.max())
    if hi == lo: hi = lo + 1.0
    counts, edges = np.histogram(s, bins=bins, range=(lo, hi))
    W, H, ML, MB, MT, MR = 640, 320, 55, 45, 28, 12
    pw, ph = W - ML - MR, H - MB - MT
    maxc = counts.max() or 1
    out = _svg_open(W, H) + [f'<text x="{ML}" y="16" font-size="13">{title}</text>'] + _axis(ML, H-MB, W-MR, H-MB)
    bw = pw / bins
    for i, c in enumerate(counts):
        h = ph * c / maxc
        out.append(f'<rect x="{ML+i*bw:.1f}" y="{H-MB-h:.1f}" width="{bw-0.5:.1f}" height="{h:.1f}" fill="{color}"/>')
    for frac in (0, .25, .5, .75, 1):
        v = lo + frac*(hi-lo)
        x = ML + frac*pw
        lab = f"1e{v:.0f}" if log_scale else f"{v:.0f}"
        out.append(f'<text x="{x-8:.1f}" y="{H-MB+14}">{lab}</text>')
    out.append(f'<text x="{ML}" y="{H-6}">{"log1p scale" if log_scale else "linear"}</text>')
    out.append("</svg>")
    Path(path).write_text("\n".join(out))

def svg_bar(labels, values, path, title="", color="#4c8bf5", fmt="{:,.0f}"):
    W, H, ML, MB, MT, MR = 640, 320, 150, 30, 28, 70
    pw, ph = W - ML - MR, H - MB - MT
    maxv = max(values) or 1
    n = len(values)
    bh = ph / n * 0.65
    out = _svg_open(W, H) + [f'<text x="{ML}" y="16" font-size="13">{title}</text>']
    for i, (lab, v) in enumerate(zip(labels, values)):
        y = MT + i*(ph/n) + (ph/n - bh)/2
        w = pw * v / maxv
        out.append(f'<rect x="{ML}" y="{y:.1f}" width="{w:.1f}" height="{bh:.1f}" fill="{color}"/>')
        out.append(f'<text x="{ML-6}" y="{y+bh/2+4:.1f}" text-anchor="end">{lab}</text>')
        out.append(f'<text x="{ML+w+5:.1f}" y="{y+bh/2+4:.1f}">{fmt.format(v)}</text>')
    out.append("</svg>")
    Path(path).write_text("\n".join(out))

def svg_scatter(x, y, path, title="", xlabel="", ylabel="", color="#e07b39", cap=4000, logx=True, logy=False):
    import random
    pts = list(zip(x.dropna(), y.dropna()))
    if len(pts) > cap:  # jitter-sample for readability; stats below use the full set
        pts = random.Random(42).sample(pts, cap)
    W, H, ML, MB, MT, MR = 640, 380, 55, 45, 28, 12
    pw, ph = W - ML - MR, H - MB - MT
    xs = np.log1p(np.array([p[0] for p in pts]).clip(min=0)) if logx else np.array([p[0] for p in pts])
    ys = np.log1p(np.array([p[1] for p in pts]).clip(min=0)) if logy else np.array([p[1] for p in pts])
    xmin, xmax = xs.min(), max(xs.max(), 1e-9)
    ymin, ymax = ys.min(), max(ys.max(), 1e-9)
    out = _svg_open(W, H) + [f'<text x="{ML}" y="16" font-size="13">{title}</text>'] + _axis(ML, H-MB, W-MR, H-MB)
    for px, py in zip(xs, ys):
        cx = ML + (px-xmin)/(xmax-xmin)*pw
        cy = H-MB - (py-ymin)/(ymax-ymin)*ph
        out.append(f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="2" fill="{color}" opacity="0.35"/>')
    out.append(f'<text x="{ML+pw/2-20}" y="{H-6}">{xlabel}</text>')
    out.append("</svg>")
    Path(path).write_text("\n".join(out))

def show_svg(name):
    from IPython.display import SVG, display
    display(SVG(filename=str(FIG_DIR / name)))
print("svg helpers ready")

## 1. Data profile — completeness & shape

First the boring-but-essential checks: missing values, duplicates, and the basic
shape of each column (skewness tells us which columns need log-scale treatment).

In [ ]:
# --- 1. completeness ---------------------------------------------------------
prof = pd.DataFrame({
    "non_null": df.notna().sum(),
    "nulls": df.isna().sum(),
    "dtype": df.dtypes.astype(str),
})
num_cols = df.select_dtypes("number").columns
prof.loc[num_cols, "median"] = df[num_cols].median()
prof.loc[num_cols, "p95"] = df[num_cols].quantile(0.95)
prof.loc[num_cols, "max"] = df[num_cols].max()
print("nulls per column:")
print(prof["nulls"])
print("\nduplicate repo_full_name:", df["repo_full_name"].duplicated().sum())
print("rows:", len(df))

In [ ]:
# --- 1b. distribution shape ---------------------------------------------------
figs = []
for c in ["opened_issues", "opened_prs", "resolved_issues", "merged_prs",
          "days_since_last_commit", "days_since_last_release", "n_contributors_top100"]:
    svg_hist(df[c], FIG_DIR / f"eda_hist_{c}.svg", title=f"{c}  (n={df[c].notna().sum():,})")
    figs.append(f"eda_hist_{c}.svg")
for c in ["bus_factor"]:
    svg_hist(df[c], FIG_DIR / f"eda_hist_{c}.svg", title=f"{c}", bins=25, log_scale=False)
    figs.append(f"eda_hist_{c}.svg")

# skew summary: which columns are long-tailed (need log treatment downstream)?
skew = df[num_cols].skew().sort_values(ascending=False)
print("skewness (|s| > 1 => heavy right tail):")
print(skew.round(2))

In [ ]:
for f in figs[:4]:
    show_svg(f)

In [ ]:
for f in figs[4:]:
    show_svg(f)

## 2. Decay-stage segmentation

The project's core question is *when does a repo start decaying*. We operationalize
"decay stage" from the maintenance signals we actually have — days since last commit
and days since last release — and cross-tabulate against activity volume to see which
combination separates healthy repos from dormant ones.

**Staging (adjustable, document in the write-up):**
- `active` — committed within 90 days
- `stale` — 91–365 days since commit
- `dormant` — > 365 days since commit (or no commits recorded)

Release recency is tracked separately because many healthy repos ship releases
infrequently; commit cadence is the stronger decay signal.

In [ ]:
# --- 2. decay stages ----------------------------------------------------------
def stage(d):
    if pd.isna(d) or d > 365: return "dormant"
    if d > 90: return "stale"
    return "active"

df["decay_stage"] = df["days_since_last_commit"].apply(stage)
df["release_lag_flag"] = (df["days_since_last_release"] > 365).fillna(True)  # no release ever => True

print(df["decay_stage"].value_counts())
print()
# activity volume by stage — the money table for the decay question
stage_cols = ["opened_issues", "opened_prs", "resolved_issues", "merged_prs"]
piv = df.groupby("decay_stage")[stage_cols + ["days_since_last_commit", "bus_factor", "n_contributors_top100"]] \
        .agg(["median", "mean"])
print(piv.round(2).T)

In [ ]:
# cross-tab: stage x release recency — are dormant repos also release-stale?
ct = pd.crosstab(df["decay_stage"], df["release_lag_flag"].map({True: "no_release_1y", False: "released_in_1y"}))
print(ct)
print()
# resolution rate (resolved / opened issues) by stage — do decaying repos stop *closing* things?
df["issue_resolution_rate"] = np.where(df["opened_issues"] > 0,
                                       df["resolved_issues"] / df["opened_issues"], np.nan)
df["pr_merge_rate"] = np.where(df["opened_prs"] > 0,
                               df["merged_prs"] / df["opened_prs"], np.nan)
print("median issue-resolution rate & PR-merge rate by stage (windowed counts):")
print(df.groupby("decay_stage")[["issue_resolution_rate", "pr_merge_rate"]].median().round(3))

In [ ]:
# visual: median activity by decay stage
labels = ["active", "stale", "dormant"]
vals = [int(df.loc[df.decay_stage == s, "opened_issues"].median() or 0) for s in labels]
svg_bar(labels, vals, FIG_DIR / "eda_stage_opened_issues.svg",
        title="median opened issues (window) by decay stage")
show_svg("eda_stage_opened_issues.svg")

## 3. Bus factor & contributor base vs. inactivity

Bus factor (contributors who did >20% of commits) is the classic *bus-factor* risk
metric. Hypothesis: as repos decay, maintenance concentrates on fewer people —
inactivity should correlate with **lower** bus factor and thinner contributor bases.

In [ ]:
# --- 3. bus factor vs inactivity ----------------------------------------------
print("median bus factor & contributors by decay stage:")
print(df.groupby("decay_stage")[["bus_factor", "n_contributors_top100"]].median().round(2))
print()
# spearman via rank (no scipy needed)
def spearman(a, b):
    m = a.notna() & b.notna()
    return a[m].rank().corr(b[m].rank())

pairs = [
    ("days_since_last_commit", "bus_factor"),
    ("days_since_last_commit", "n_contributors_top100"),
    ("days_since_last_release", "bus_factor"),
    ("opened_issues", "n_contributors_top100"),
    ("merged_prs", "bus_factor"),
]
for x, y in pairs:
    print(f"spearman({x}, {y}) = {spearman(df[x], df[y]):+.3f}")

In [ ]:
svg_scatter(df["days_since_last_commit"], df["bus_factor"], FIG_DIR / "eda_busfactor_vs_inactivity.svg",
            title="bus factor vs days since last commit (sampled)",
            xlabel="days since last commit")
show_svg("eda_busfactor_vs_inactivity.svg")

## 4. Repo archetypes — unsupervised grouping

A tiny from-scratch k-means (k=4) on standardized, log-transformed activity features.
No sklearn in this env; the point is descriptive — do distinct *shapes* of repo
health emerge? We compare archetype centroids against the decay stages to check that
the clusters are meaningful rather than arbitrary.

In [ ]:
# --- 4. k-means (from scratch, deterministic seed) ----------------------------
feats = ["opened_issues", "opened_prs", "resolved_issues", "merged_prs",
         "days_since_last_commit", "bus_factor", "n_contributors_top100"]
# clip(lower=0): a handful of repos have *negative* days_since_* (a tz quirk in the
# collector — see section 5 note). log1p(-1) = -inf would otherwise poison standardization.
X = np.log1p(df[feats].fillna(0).clip(lower=0))
X = (X - X.mean()) / X.std(ddof=0)
Xv = X.values

rng = np.random.default_rng(42)
k = 4
C = Xv[rng.choice(len(Xv), k, replace=False)].copy()
for _ in range(60):
    d2 = ((Xv[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    lab = d2.argmin(axis=1)
    newC = np.array([Xv[lab == j].mean(axis=0) if (lab == j).any() else C[j] for j in range(k)])
    if np.allclose(newC, C): break
    C = newC

df["archetype"] = lab
print("cluster sizes:")
print(pd.Series(lab).value_counts().sort_index())

In [ ]:
# what does each archetype look like? (raw medians, not standardized)
arch = df.groupby("archetype")[feats].median()  # rows = archetypes
dom_stage = df.groupby("archetype")["decay_stage"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan)
arch["dominant_decay_stage"] = dom_stage
print(arch.round(1))
print()
# archetype x decay-stage overlap — are clusters aligned with health?
print(pd.crosstab(df["archetype"], df["decay_stage"]))

In [ ]:
# scatter: two dominant standardized features, colored by archetype (manual palette)
pal = ["#4c8bf5", "#e07b39", "#5cb85c", "#d63384"]
W, H, ML, MB, MT, MR = 640, 420, 55, 45, 28, 12
pw, ph = W - ML - MR, H - MB - MT
i1, i2 = feats.index("days_since_last_commit"), feats.index("merged_prs")
out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}" font-family="monospace" font-size="11">']
out.append(f'<text x="{ML}" y="16" font-size="13">archetypes: log-merged_prs vs log-days_since_last_commit</text>')
import random
idx = list(range(len(Xv)))
random.Random(7).shuffle(idx)
for j in idx[:4000]:
    x = Xv[j, i2]; y = Xv[j, i1]
    cx = ML + (x - Xv[:, i2].min()) / max(float(Xv[:, i2].max() - Xv[:, i2].min()), 1e-9) * pw
    cy = H - MB - (y - Xv[:, i1].min()) / max(float(Xv[:, i1].max() - Xv[:, i1].min()), 1e-9) * ph
    out.append(f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="2" fill="{pal[lab[j]]}" opacity="0.4"/>')
out.append("</svg>")
(FIG_DIR / "eda_archetypes.svg").write_text("\n".join(out))
show_svg("eda_archetypes.svg")

## 5. Summary & derived dataset

Key findings so far (fill in the numbers from the cells above in the write-up):

1. **Which columns are long-tailed** and need log treatment downstream.
2. **How activity volume collapses across decay stages** — the central evidence for
   the decay signal.
3. **Whether bus factor / contributor base thins as repos go dormant.**
4. **Whether k-means archetypes align with the decay staging** (i.e., health is a
   low-dimensional structure, not noise).

**Data-quality flag (for the collection notebook):** 46 repos show *negative*
`days_since_last_commit` and 8 negative `days_since_last_release`. The host clock runs
UTC-04:00 while GitHub timestamps are UTC, so commits/releases made within the last few
hours compute as slightly "in the future." This analysis clips at 0; the collector
should clamp `days_since_* = max(0, ...)` (or compute in UTC) — **fix after the current
run finishes, don't restart it for this.** The affected rows are a <1% edge case and
the sign is meaningless either way.

The enriched snapshot (original columns + `decay_stage`, rates, `archetype`) is
exported for the integration notebook — written to its own file so the live
collection CSV is never modified.

In [ ]:
# --- 5. export enriched snapshot ----------------------------------------------
out_cols = list(df.columns)
df.to_csv(ENRICHED_CSV, index=False)
print(f"wrote {ENRICHED_CSV.name}: {len(df):,} rows x {len(out_cols)} cols")
print("new derived columns:", [c for c in out_cols if c not in pd.read_csv(SNAP_CSV).columns])